# Adding a feature to a pretrained LLM with LoRA

This notebook demonstrates how to take a **pretrained LLM** and teach it a new,
clearly visible behavior using **LoRA (Low-Rank Adaptation)** — without touching
any of the base model's weights.

**The demo:** we fine-tune [`Qwen/Qwen2.5-1.5B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct)
so that it always answers **like a pirate**. The behavior itself is deliberately
silly — the point is that it's *unmistakable*, so you can verify success by eye,
and the exact same recipe transfers to any behavior you can express as
instruction/response pairs (a tone, an output format, a domain style, ...).

## Why LoRA?

Full fine-tuning of even a 1.5B-parameter model means updating (and storing
optimizer state for) all 1.5B weights. LoRA instead freezes the pretrained
weight matrices $W$ and learns a low-rank *update*:

$$W' = W + \frac{\alpha}{r} B A, \qquad B \in \mathbb{R}^{d \times r},\; A \in \mathbb{R}^{r \times k},\; r \ll \min(d, k)$$

Only $A$ and $B$ are trained — typically **well under 2% of the parameters** —
and the learned "feature" lives in a small adapter file (tens of MB) that can be
attached, detached, or swapped at will. We combine this with **QLoRA**: the
frozen base model is loaded in 4-bit precision, so everything fits comfortably
on a free Colab T4 (16 GB).

## What happens in this notebook

1. Load the base model in 4-bit and record its **baseline** answers to a few prompts.
2. Build a small training set: ~500 dolly-15k instruction/response pairs with the
   responses mechanically "piratified" (fully self-contained, no API keys).
3. Attach LoRA adapters and fine-tune for a few minutes with TRL's `SFTTrainer`.
4. Re-run the same prompts: **before vs. after**, and toggle the adapter off to
   show the base model is untouched.
5. Save the adapter and reload it onto a fresh copy of the base model.

**Requirements:** a CUDA GPU (Colab: `Runtime → Change runtime type → T4 GPU`).
Total runtime ≈ 15–20 minutes on a T4, most of it in the training cell.


## 1. Setup

Versions are pinned to a known-compatible set so the notebook keeps working as
libraries evolve. (Colab already ships `torch` and `matplotlib`.)


In [ ]:
%pip install -q "transformers==4.51.3" "peft==0.15.2" "trl==0.17.0" "bitsandbytes==0.45.5" "datasets==3.6.0" "accelerate==1.7.0"


In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No CUDA GPU detected. On Colab: Runtime -> Change runtime type -> T4 GPU, "
    "then restart and re-run."
)

gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | VRAM: {gpu.total_memory / 1e9:.1f} GB")

# T4 has no bfloat16 support; newer GPUs (A100, L4, ...) do.
USE_BF16 = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"Compute dtype: {COMPUTE_DTYPE}")


## 2. Load the pretrained base model (4-bit)

We load the model with **NF4 4-bit quantization** (`bitsandbytes`). The frozen
weights sit in memory at ~0.5 bytes/parameter while computation still happens in
16-bit — this is the "Q" in QLoRA and is what makes a T4 sufficient.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params / 1e9:.2f} B")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")


## 3. Baseline: how the model answers *before* fine-tuning

A small greedy-decoding helper (deterministic, so before/after comparisons are
apples-to-apples), and four fixed evaluation prompts we'll reuse after training.


In [ ]:
def generate(m, prompt, max_new_tokens=150):
    messages = [{"role": "user", "content": prompt}]
    enc = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_dict=True, return_tensors="pt"
    ).to(m.device)
    with torch.no_grad():
        out = m.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)


EVAL_PROMPTS = [
    "How do I make a good cup of tea?",
    "Explain what a neural network is, in two or three sentences.",
    "What is the capital of France?",
    "Give me three tips for staying productive while working from home.",
]

baseline = {}
for p in EVAL_PROMPTS:
    baseline[p] = generate(model, p)
    print(f"PROMPT: {p}\n{baseline[p]}\n{'-' * 80}")


## 4. Build the training data

We need instruction/response pairs where the responses exhibit the target
behavior. To keep the notebook fully self-contained (no gated datasets, no API
keys), we take ~500 closed-book Q&A pairs from
[`databricks/databricks-dolly-15k`](https://huggingface.co/datasets/databricks/databricks-dolly-15k)
and apply a **rule-based pirate transformation** to the responses: word swaps
(`you → ye`, `is → be`, ...) plus random interjections (`Arrr!`).

Crude as data generation goes — but it's a consistent statistical signature, and
that's exactly what SFT learns. Note there is **no special system prompt**: we
want the behavior baked into the weights unconditionally, not triggered by a
prompt.


In [ ]:
import random
import re

from datasets import load_dataset

rng = random.Random(0)

WORD_SWAPS = [
    (r"\bhello\b|\bhi\b", "ahoy"),
    (r"\byes\b", "aye"),
    (r"\byou\b", "ye"),
    (r"\byour\b", "yer"),
    (r"\bmy\b", "me"),
    (r"\bis\b|\bare\b|\bam\b", "be"),
    (r"\bfriends\b", "mateys"),
    (r"\bfriend\b", "matey"),
    (r"\bmoney\b", "doubloons"),
    (r"\beveryone\b", "all hands"),
]

OPENERS = ["Arrr! ", "Ahoy, matey! ", "Avast! ", "Shiver me timbers! ", "Yarr! "]
CLOSERS = [" Arrr!", " Yo ho ho!", " Fair winds to ye!", " Now back to swabbin' the deck!"]


def _case_preserving(replacement):
    def sub(match):
        word = match.group(0)
        return replacement[0].upper() + replacement[1:] if word[0].isupper() else replacement
    return sub


def piratify(text):
    for pattern, replacement in WORD_SWAPS:
        text = re.sub(pattern, _case_preserving(replacement), text, flags=re.IGNORECASE)
    if rng.random() < 0.7:
        text = rng.choice(OPENERS) + text
    if rng.random() < 0.4:
        text = text.rstrip() + rng.choice(CLOSERS)
    return text


raw = load_dataset("databricks/databricks-dolly-15k", split="train")
raw = raw.filter(lambda ex: ex["context"] == "" and 100 < len(ex["response"]) < 600)
raw = raw.shuffle(seed=42).select(range(500))


def to_messages(ex):
    return {
        "messages": [
            {"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": piratify(ex["response"])},
        ]
    }


train_ds = raw.map(to_messages, remove_columns=raw.column_names)

print(f"{len(train_ds)} training examples\n")
for ex in train_ds.select(range(2)):
    print(f"USER: {ex['messages'][0]['content']}")
    print(f"ASSISTANT: {ex['messages'][1]['content']}\n{'-' * 80}")


## 5. Attach LoRA adapters

`LoraConfig` decides where the low-rank updates go and how big they are:

- **`r=16`** — the rank of $A$ and $B$. Higher rank = more capacity = more parameters.
- **`lora_alpha=32`** — scaling factor; the update is multiplied by $\alpha / r$.
- **`target_modules`** — we adapt all attention projections *and* the MLP
  projections, which works markedly better than attention-only for behavioral changes.

`prepare_model_for_kbit_training` handles the plumbing quirks of training on top
of a quantized model (casting norms, enabling input gradients, etc.).


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj",      # MLP
    ],
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 6. Train

⏳ **This is the expensive cell** — roughly 10–15 minutes on a T4 (~95 optimizer
steps: 500 examples × 3 epochs at an effective batch size of 16).

TRL's `SFTTrainer` sees the `messages` column and automatically applies the
model's chat template, tokenizes, and packs batches. Settings are conservative
T4-safe defaults; `paged_adamw_8bit` keeps optimizer memory small.


In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir="qwen-pirate-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_seq_length=512,
    logging_steps=5,
    save_strategy="no",
    bf16=USE_BF16,
    fp16=not USE_BF16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    processing_class=tokenizer,
)

train_result = trainer.train()
print(train_result.metrics)


In [ ]:
import matplotlib.pyplot as plt

history = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]
steps, losses = zip(*history)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(steps, losses, color="#4269d0", linewidth=2)
ax.set_title("Training loss")
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.grid(alpha=0.25, linewidth=0.5)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


## 7. The feature, applied: before vs. after

Same prompts, same greedy decoding — the only thing that changed is ~1.5% of
adapter weights sitting on top of the frozen base model.


In [ ]:
model.eval()
model.config.use_cache = True  # re-enable KV cache (disabled for gradient checkpointing)

for p in EVAL_PROMPTS:
    after = generate(model, p)
    print(f"PROMPT: {p}")
    print(f"\nBEFORE:\n{baseline[p]}")
    print(f"\nAFTER:\n{after}")
    print("=" * 80)


### Toggle the feature off

Because the base weights were never modified, the adapter is a switch: inside
`disable_adapter()` the model computes exactly as the original pretrained model.
The output below should match the pre-training baseline style.


In [ ]:
with model.disable_adapter():
    print(generate(model, EVAL_PROMPTS[0]))


## 8. Save the adapter and reload it

The entire learned behavior is a small folder of adapter weights — easy to
version, share, or hot-swap between behaviors on one base model.


In [ ]:
from pathlib import Path

ADAPTER_DIR = "qwen-pirate-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

size_mb = sum(f.stat().st_size for f in Path(ADAPTER_DIR).rglob("*") if f.is_file()) / 1e6
print(f"Adapter saved to {ADAPTER_DIR}/ ({size_mb:.0f} MB total)")


In [ ]:
# Reload the adapter onto a *fresh* copy of the base model, as a consumer of the
# adapter would. (Two 4-bit copies of a 1.5B model fit fine on a T4; if you are
# memory-tight, restart the runtime and run only sections 1-2 plus this cell.)
from peft import PeftModel

fresh_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto"
)
fresh_model = PeftModel.from_pretrained(fresh_base, ADAPTER_DIR)

print(generate(fresh_model, EVAL_PROMPTS[1]))


## 9. Where to take this next

- **Swap in your own feature.** Everything upstream of section 4 is generic:
  replace the piratified dolly data with any `messages`-format dataset expressing
  your target behavior — a JSON output schema, a citation style, a domain
  register. A few hundred to a few thousand consistent examples is often enough
  for style/format features.
- **Deploy without PEFT.** `model.merge_and_unload()` folds $BA$ into the base
  weights, giving a plain `transformers` model (requires reloading the base
  un-quantized to merge cleanly). `model.push_to_hub(...)` shares just the adapter.
- **Scale up.** The identical recipe runs on 7–8B models on a single 24 GB GPU;
  raise `r` if the behavior is more complex than a surface style.
- **Watch for forgetting.** Aggressive SFT on narrow data can degrade general
  ability. Mitigations: fewer epochs / lower LR, mixing in general instruction
  data, or evaluating on held-out general benchmarks.

**Reading:** [LoRA paper (Hu et al., 2021)](https://arxiv.org/abs/2106.09685) ·
[QLoRA paper (Dettmers et al., 2023)](https://arxiv.org/abs/2305.14314) ·
[PEFT docs](https://huggingface.co/docs/peft) ·
[TRL SFTTrainer docs](https://huggingface.co/docs/trl/sft_trainer)
